# AI & Data Jobs Market Explorer
> **36K job listings, 280 companies, 318K job-skill edges** | [Dataset](https://www.kaggle.com/datasets/lorenzoscaturchio/ai-data-jobs-skills-salaries-2024-2026)

**Files:** `jobs.csv`, `companies.csv`, `job_skills.csv`, `salary_benchmarks.csv`, `skill_demand_monthly.csv`

## Table of Contents
1. [Objective](#objective)
2. [Setup & Loading](#setup)
3. [Dataset Inventory](#inventory)
4. [Role and Salary Landscape](#roles)
5. [Skill Demand and Trend Shifts](#skills)
6. [Benchmarks and Remote Mix](#benchmarks)
7. [Key Findings](#findings)


## 1. Objective <a id='objective'></a>

This notebook is a compact first-pass explorer for the jobs-market dataset. The goal is to make the five linked tables immediately usable for:

- salary modeling and benchmarking
- skill-demand trend analysis
- job-to-skill graph workflows
- role mix analysis across 2024-2026

The emphasis here is on quick orientation and modeling-ready hypotheses rather than exhaustive EDA.


In [ ]:
TARGET_COL = 'salary_mid_usd'
MODELING_TASK = 'multi-table exploration with regression, trend analysis, and ranking use cases'
PRIMARY_METRIC = 'depends on downstream task (RMSE / MAE for salary, AUC / MAP for ranking, descriptive trend checks for EDA)'
VALIDATION_PLAN = 'start with time-aware splits for jobs and grouped joins for skill edges'

print('Explorer framing')
print('-' * 70)
print(f'Target candidate   : {TARGET_COL}')
print(f'Primary task       : {MODELING_TASK}')
print(f'Validation default : {VALIDATION_PLAN}')
print(f'Primary metric     : {PRIMARY_METRIC}')


## 2. Setup & Loading <a id='setup'></a>


In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
sns.set_palette('deep')

candidate_dirs = [
    Path('/kaggle/input/ai-data-jobs-skills-salaries-2024-2026'),
    Path('/kaggle/input/ai-data-jobs-market'),
    Path('.'),
]

input_root = Path('/kaggle/input')
if input_root.exists():
    discovered = [p for p in input_root.iterdir() if p.is_dir() and (p / 'jobs.csv').exists() and (p / 'companies.csv').exists()]
    candidate_dirs = discovered + candidate_dirs

DATA_DIR = next((p for p in candidate_dirs if (p / 'jobs.csv').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Could not locate jobs-market CSV files in /kaggle/input or the working directory.')

jobs = pd.read_csv(DATA_DIR / 'jobs.csv')
companies = pd.read_csv(DATA_DIR / 'companies.csv')
job_skills = pd.read_csv(DATA_DIR / 'job_skills.csv')
salary_benchmarks = pd.read_csv(DATA_DIR / 'salary_benchmarks.csv')
skill_demand_monthly = pd.read_csv(DATA_DIR / 'skill_demand_monthly.csv')

print(f'Data directory        : {DATA_DIR}')
print(f'jobs                 : {jobs.shape[0]:,} rows x {jobs.shape[1]} columns')
print(f'companies            : {companies.shape[0]:,} rows x {companies.shape[1]} columns')
print(f'job_skills           : {job_skills.shape[0]:,} rows x {job_skills.shape[1]} columns')
print(f'salary_benchmarks    : {salary_benchmarks.shape[0]:,} rows x {salary_benchmarks.shape[1]} columns')
print(f'skill_demand_monthly : {skill_demand_monthly.shape[0]:,} rows x {skill_demand_monthly.shape[1]} columns')


## 3. Dataset Inventory <a id='inventory'></a>


In [ ]:
file_inventory = pd.DataFrame([
    {'table': 'jobs', 'rows': len(jobs), 'columns': jobs.shape[1], 'grain': 'one row per job listing'},
    {'table': 'companies', 'rows': len(companies), 'columns': companies.shape[1], 'grain': 'one row per company'},
    {'table': 'job_skills', 'rows': len(job_skills), 'columns': job_skills.shape[1], 'grain': 'one row per job-skill edge'},
    {'table': 'salary_benchmarks', 'rows': len(salary_benchmarks), 'columns': salary_benchmarks.shape[1], 'grain': 'grouped benchmark slice'},
    {'table': 'skill_demand_monthly', 'rows': len(skill_demand_monthly), 'columns': skill_demand_monthly.shape[1], 'grain': 'month-skill aggregate'},
])

display(file_inventory)
print('Core jobs columns')
display(jobs[['job_title', 'seniority', 'country', 'salary_mid_usd', 'remote_type', 'ai_focus_area', 'required_skills']].head())

print('Top job titles')
display(jobs['job_title'].value_counts().head(10).rename_axis('job_title').reset_index(name='postings'))


## 4. Role and Salary Landscape <a id='roles'></a>


In [ ]:
role_counts = jobs['job_title'].value_counts().head(10).sort_values()
role_salary = jobs.groupby('job_title', as_index=False)['salary_mid_usd'].median().sort_values('salary_mid_usd', ascending=False).head(10)
country_salary = jobs.groupby('country', as_index=False)['salary_mid_usd'].median().sort_values('salary_mid_usd', ascending=False).head(10)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
role_counts.plot(kind='barh', ax=axes[0], title='Top job titles by posting volume')
axes[0].set_xlabel('Postings')
axes[0].set_ylabel('')

sns.barplot(data=role_salary, x='salary_mid_usd', y='job_title', ax=axes[1])
axes[1].set_title('Median salary by role')
axes[1].set_xlabel('Median salary_mid_usd')
axes[1].set_ylabel('')

sns.barplot(data=country_salary, x='salary_mid_usd', y='country', ax=axes[2])
axes[2].set_title('Median salary by country')
axes[2].set_xlabel('Median salary_mid_usd')
axes[2].set_ylabel('')

plt.tight_layout()
plt.show()

salary_snapshot = jobs.groupby(['job_title', 'seniority'], as_index=False)['salary_mid_usd'].median()
display(salary_snapshot.sort_values('salary_mid_usd', ascending=False).head(12))


## 5. Skill Demand and Trend Shifts <a id='skills'></a>


In [ ]:
jobs['posted_month'] = pd.to_datetime(jobs['posted_date']).dt.to_period('M').astype(str)
role_mix = (
    jobs[jobs['job_title'].isin(['AI Engineer', 'LLM Engineer', 'ML Engineer', 'Data Scientist'])]
    .groupby(['posted_month', 'job_title'])['job_id']
    .count()
    .reset_index(name='postings')
)
role_pivot = role_mix.pivot(index='posted_month', columns='job_title', values='postings').fillna(0)
role_share = role_pivot.div(role_pivot.sum(axis=1), axis=0)

latest_skills = (
    skill_demand_monthly[skill_demand_monthly['year_month'] == skill_demand_monthly['year_month'].max()]
    .sort_values('job_count', ascending=False)
    .head(12)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 5))
role_share[['AI Engineer', 'LLM Engineer', 'ML Engineer', 'Data Scientist']].plot(ax=axes[0])
axes[0].set_title('Role share shift over time')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Share of tracked roles')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(data=latest_skills, x='job_count', y='skill', ax=axes[1])
axes[1].set_title(f"Top skills in {skill_demand_monthly['year_month'].max()}")
axes[1].set_xlabel('Job count')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

display(latest_skills[['skill', 'skill_category', 'job_count', 'median_salary_mid_usd', 'remote_share', 'share_of_postings']])


## 6. Benchmarks and Remote Mix <a id='benchmarks'></a>


In [ ]:
remote_mix = jobs['remote_type'].value_counts(normalize=True).rename('share').mul(100).round(1)
country_remote = (
    jobs.assign(is_remote=(jobs['remote_type'] == 'remote').astype(int))
    .groupby('country', as_index=False)
    .agg(remote_share=('is_remote', 'mean'), median_salary=('salary_mid_usd', 'median'), postings=('job_id', 'count'))
    .sort_values(['postings', 'median_salary'], ascending=[False, False])
)

print('Remote mix (%)')
display(remote_mix.reset_index().rename(columns={'index': 'remote_type'}))

print('Top benchmark slices by median salary')
display(salary_benchmarks.sort_values('salary_median_usd', ascending=False).head(12))

print('Country remote-share view')
display(country_remote.head(12))


## 7. Key Findings <a id='findings'></a>

- The dataset is strong for **salary modeling** because compensation moves cleanly with seniority, geography, company size, and role family.
- `job_skills.csv` makes the package immediately useful for **graph ML**, retrieval, and skill recommendation work instead of forcing skill parsing from raw text.
- The monthly aggregate tables make it easy to study the 2024-2026 shift toward **AI Engineer** and **LLM Engineer** roles without rebuilding time-based features from scratch.
- `salary_benchmarks.csv` is already presentation-ready for dashboards and compensation benchmarking workflows.
- A good next modeling baseline is a time-aware salary regressor on `jobs.csv`, then a skill-demand ranking or forecasting task using the monthly trend table.
